# Hardwood Biochar at 400 °C — Five-Molecule Series

Generates five structurally distinct biochar molecules, all conditioned on the
UC Davis temperature × feedstock model at **T = 400 °C, feedstock = hardwood**.
Composition targets (H/C, O/C, aromaticity) are derived automatically; the
five molecules differ in carbon count and topological disorder.

**Output** — for each molecule, separate `.gro` and `.itp` files are written
to `output/hardwood_400C/`:

| Label | Carbons | Defect fraction | `.gro` | `.itp` |
|-------|--------:|----------------:|--------|--------|
| H4A | 45 | 0 % | `bc400_A.gro` | `bc400_A.itp` |
| H4B | 60 | 0 % | `bc400_B.gro` | `bc400_B.itp` |
| H4C | 75 | 5 % | `bc400_C.gro` | `bc400_C.itp` |
| H4D | 90 | 10 % | `bc400_D.gro` | `bc400_D.itp` |
| H4E | 110 | 15 % | `bc400_E.gro` | `bc400_E.itp` |

> **Note on validation mode** — 400 °C biochar empirically has H/C ≈ 0.54
> (influenced by non-aromatic H in real char). Pure aromatic PAH sheets of
> this size achieve H/C ≈ 0.35–0.50 because edge-H sites scale as the
> perimeter, not the area. `strict=False` is used here so generation succeeds;
> GROMACS energy minimisation resolves any minor peri-H clashes in the output
> files.

**Requirements** (in addition to `biochar`):
```bash
conda install -c conda-forge matplotlib pandas rdkit
```

In [ ]:
import logging
from pathlib import Path

import pandas as pd
from IPython.display import display
from rdkit.Chem.Draw import MolsToGridImage

from biochar import BiocharGenerator, GeneratorConfig
from biochar.temperature_model import get_default_model

# Show INFO logs so generation progress is visible in the notebook
logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(message)s")

## 1 · Temperature-model composition preview

Inspect the UC Davis model prediction for 400 °C hardwood before generating
any structures. These targets are injected into every molecule below.

In [ ]:
model = get_default_model()
comp_preview = model.composition(temperature=400, feedstock="hardwood")

print("Predicted composition — 400 °C hardwood biochar")
print("=" * 50)
for key, val in comp_preview.items():
    print(f"  {key:<26s} {val:.4f}")

## 2 · Molecule configurations

All five share `temperature=400, feedstock="hardwood"`.  Structural diversity
comes from varying `target_num_carbons`, `defect_fraction`, and `seed`.

In [ ]:
OUTPUT_DIR = Path("output/hardwood_400C")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SPECS = [
    dict(molecule_name="H4A", target_num_carbons=45,  defect_fraction=0.00, seed=11, basename="bc400_A", description="Small, pure hexagonal"),
    dict(molecule_name="H4B", target_num_carbons=60,  defect_fraction=0.00, seed=22, basename="bc400_B", description="Medium, pure hexagonal"),
    dict(molecule_name="H4C", target_num_carbons=75,  defect_fraction=0.05, seed=33, basename="bc400_C", description="Medium-large, 5 % pentagons"),
    dict(molecule_name="H4D", target_num_carbons=90,  defect_fraction=0.10, seed=44, basename="bc400_D", description="Large, 10 % pentagons"),
    dict(molecule_name="H4E", target_num_carbons=110, defect_fraction=0.15, seed=55, basename="bc400_E", description="X-large, 15 % pentagons"),
]

print(f"Output directory: {OUTPUT_DIR.resolve()}")
print(f"Molecules to generate: {[s['molecule_name'] for s in SPECS]}")

## 3 · Generate and export

For each molecule, the pipeline runs:
1. PAH carbon skeleton (hexagonal + optional pentagon defects)
2. Oxygen functional groups + hydrogen saturation
3. 3-D geometry (RDKit ETKDGv3 or hex-lattice for larger sheets)
4. OPLS-AA typing and charge assignment
5. Export to `.gro`, `.itp`, `.top`

In [ ]:
entries = []

for spec in SPECS:
    print(f"\n{'─' * 60}")
    print(f"Generating {spec['molecule_name']}  ({spec['description']})")
    print(f"{'─' * 60}")

    config = GeneratorConfig(
        target_num_carbons=spec["target_num_carbons"],
        defect_fraction=spec["defect_fraction"],
        temperature=400,
        feedstock="hardwood",
        molecule_name=spec["molecule_name"],
        seed=spec["seed"],
        # strict=False: PAH sheets at low pyrolysis T cannot achieve the
        # empirical H/C=0.54 (edge-H scales as perimeter, not area). Structures
        # are still valid for MD; GROMACS em resolves any peri-H clashes.
        strict=False,
    )

    gen = BiocharGenerator(config)
    mol, coords, composition = gen.generate()
    gen.print_summary()

    gro_path, top_path, itp_path = gen.export_gromacs(
        output_directory=str(OUTPUT_DIR),
        basename=spec["basename"],
    )

    entries.append(dict(spec=spec, mol=mol, coords=coords,
                        composition=composition, ring_composition=gen.ring_composition,
                        gro_path=gro_path, itp_path=itp_path))

print("\n" + "=" * 60)
print(f"Done — {len(entries)} molecules generated.")
print("=" * 60)

## 4 · Summary table

In [ ]:
def aromaticity_pct(mol):
    """Fraction of carbon atoms that are aromatic, as a percentage."""
    carbons = [a for a in mol.GetAtoms() if a.GetAtomicNum() == 6]
    if not carbons:
        return 0.0
    aromatic = sum(1 for a in carbons if a.GetIsAromatic())
    return round(100 * aromatic / len(carbons), 1)


rows = []
for e in entries:
    c = e["composition"]
    rc = e["ring_composition"] or {}
    rows.append({
        "Name": e["spec"]["molecule_name"],
        "Description": e["spec"]["description"],
        "Formula": c.molecular_formula,
        "MW (g/mol)": round(c.molecular_weight, 1),
        "H/C (actual)": round(c.H_C_ratio, 3),
        "O/C (actual)": round(c.O_C_ratio, 3),
        "Arom. C %": aromaticity_pct(e["mol"]),
        "Hexagons": rc.get("hexagons", "—"),
        "Pentagons": rc.get("pentagons", 0),
        ".gro file": e["gro_path"].name,
        ".itp file": e["itp_path"].name,
    })

df = pd.DataFrame(rows)
display(df.style.hide(axis="index"))

## 5 · 2-D structure grid

In [ ]:
mols = [e["mol"] for e in entries]
legends = [
    (
        f"{e['spec']['molecule_name']} · {e['composition'].num_carbons}C · "
        f"def={e['spec']['defect_fraction']:.0%}\n"
        f"H/C={e['composition'].H_C_ratio:.3f}  O/C={e['composition'].O_C_ratio:.3f}"
    )
    for e in entries
]

img = MolsToGridImage(
    mols,
    molsPerRow=3,
    subImgSize=(500, 360),
    legends=legends,
)
display(img)

## 6 · Generated file listing

In [ ]:
print(f"Files written to {OUTPUT_DIR.resolve()}:\n")
print(f"{'Name':<6}  {'GRO file':<22}  {'size':>8}    {'ITP file':<22}  {'size':>8}")
print("─" * 75)
for e in entries:
    gp = e["gro_path"]
    ip = e["itp_path"]
    print(
        f"{e['spec']['molecule_name']:<6}  {gp.name:<22}  {gp.stat().st_size:>7,} B"
        f"    {ip.name:<22}  {ip.stat().st_size:>7,} B"
    )

## 7 · GROMACS usage

Each `.gro` + `.itp` pair is ready to use in a GROMACS simulation.

**Single molecule — energy minimisation:**
```bash
gmx editconf -f output/hardwood_400C/bc400_C.gro -o box.gro -bt cubic -d 1.5
gmx solvate  -cp box.gro -cs spc216.gro -o solvated.gro -p bc400_C.top
gmx grompp   -f em.mdp -c solvated.gro -p bc400_C.top -o em.tpr
gmx mdrun    -v -deffnm em
```

**Mixed system — include all five ITP files in a combined topology:**
```
; combined.top
#include "gromos54a7.ff/forcefield.itp"
#include "output/hardwood_400C/bc400_A.itp"
#include "output/hardwood_400C/bc400_B.itp"
#include "output/hardwood_400C/bc400_C.itp"
#include "output/hardwood_400C/bc400_D.itp"
#include "output/hardwood_400C/bc400_E.itp"

[ system ]
Hardwood 400 C biochar — 5-molecule ensemble

[ molecules ]
H4A  1
H4B  1
H4C  1
H4D  1
H4E  1
```